# VESM: Getting Started (Quickstart & Inference)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ntranoslab/vesm/blob/main/notebooks/VESM_Getting_Started.ipynb)

### Requirements
- Python 3.9+ (recommended), `torch`, `transformers`, `huggingface_hub`, `numpy`, `matplotlib`, `seaborn`
- GPU is optional but recommended for speed

## Table of contents
1. [Setup & Imports](#setup--imports)
2. [Load a VESM Model](#load-a-model)
3. [Run Inference & Get Scores](#vesm_inference)
4. [Visualize Results](#visualize-results)  
5. [Download Prediction Scores](#download-prediction-scores)
6. [VESM3 Inference](#vesm3-inference)

<a id="setup--imports"></a>

## 1. Setup & Imports

In [1]:
import torch
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, EsmForMaskedLM
import numpy as np
import pandas as pd
import warnings
warnings.simplefilter("ignore", FutureWarning) # ignore future warnings from transformers package
warnings.simplefilter("ignore", UserWarning) # ignore user warnings

In [2]:

local_dir = 'vesm' # local directory to store models

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    device = torch.device('cuda:0')
else:
    device = 'cpu'

CUDA available: True
GPU name: Tesla T4


<a id="load-a-model"></a>

## 2. Load VESM model

In [3]:
esm_dict = {
    "VESM_3B": 'facebook/esm2_t36_3B_UR50D',
    "VESM_650M": 'facebook/esm2_t33_650M_UR50D',
    "VESM_150M": 'facebook/esm2_t30_150M_UR50D',
    "VESM_35M": 'facebook/esm2_t12_35M_UR50D',
    "VESM3": "esm3_sm_open_v1"
}

def load_vesm(model_name="VESM_3B", local_dir="vesm", device='cuda'):
    if model_name in esm_dict:
        ckt = esm_dict[model_name]
    else:
        print("Model not found")
        return None
    # download weights
    hf_hub_download(repo_id="ntranoslab/vesm", filename=f"{model_name}.pth", local_dir=local_dir)
    # load base model
    if model_name == "VESM3":
      from esm.models.esm3 import ESM3
      model = ESM3.from_pretrained(ckt, device=device).to(torch.float)
      tokenizer = model.tokenizers.sequence
    else:
      model = EsmForMaskedLM.from_pretrained(ckt).to(device)
      tokenizer = AutoTokenizer.from_pretrained(ckt)
    # load pretrained VESM
    model.load_state_dict(torch.load(f'{local_dir}/{model_name}.pth'), strict=False)
    return model, tokenizer

We first load the VESM_3B checkpoint from Hugging Face

In [4]:
model_name = 'VESM_3B'
model, tokenizer = load_vesm(model_name, local_dir=local_dir, device=device)
sequence_vocabs = tokenizer.get_vocab()

VESM_3B.pth: reconstructing file:   0%|          |  0.00B /  971MB            

VESM_3B.pth: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/58.0k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/588 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

# VESM Inference

The following function is to get log-likelihood ratio (LLR) scores for all possible single missense mutations of a given sequence.

In [7]:
def get_LLR(sequence, esm_model, device='cuda'):
  """
    @param sequence: str, input protein sequence
    @param esm_model: loaded VESM model
    @param device: 'cuda' or 'cpu'
    @return: pd.DataFrame, LLR scores for all missense mutations
  """
  tokens = tokenizer(sequence, return_tensors='pt')
  batch_tokens = tokens['input_ids']
  esm_model.eval()
  with torch.no_grad():
    logits =torch.log_softmax(esm_model(batch_tokens.to(device),)['logits'],dim=-1)[0,:,:].cpu()
  tok = batch_tokens[0,:].cpu()
  wt_norm = logits[np.arange(len(tok)), tok].unsqueeze(1)
  LLR = logits - wt_norm
  LLR=LLR[1:-1,4:24].numpy()
  AAorder=['K','R','H','E','D','N','Q','T','S','C','G','A','V','L','I','M','P','Y','F','W']
  LLR_ = pd.DataFrame(LLR,columns=tokenizer.all_tokens[4:24],index=list(sequence)).T.loc[AAorder]
  LLR_.columns = [j.split('.')[0]+' '+str(i+1) for i,j in enumerate(LLR_.columns)]
  return LLR_

<a id="vesm_inference"></a>
## 3. Inference on a protein sequence

In [10]:
!pip install biopython ete3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 17.5 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.0 MB/s  0:00:00
  Created wheel for ete3: filename=ete3-3.1.3-py3-none-any.whl size=2273899 sha256=d522d02504997a2207435e4e2c1972ac4d8036beb9367a0aecee2d083aa9a6db
  Stored in directory: /root/.cache/pip/wheels/88/cc/c1/0e0d50d8a29caccf5d482d883af1bc9d9af5b043ea108008c8
Successfully built ete3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [biopython]


In [13]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
from Bio import SeqIO

fasta_path = '/content/drive/MyDrive/joint_asr/MTOR_ox1z4f7_/joint.fa'

# Load and print the sequences
sequences = list(SeqIO.parse(fasta_path, 'fasta'))
print(f"Successfully loaded {len(sequences)} sequences from {fasta_path}.")
for record in sequences[:5]:
    print(f"ID: {record.id}")
    print(f"Sequence: {record.seq[:50]}...")
    print(f"Length: {len(record.seq)}\n")

Successfully loaded 934 sequences from /content/drive/MyDrive/joint_asr/MTOR_ox1z4f7_/joint.fa.
ID: hg38
Sequence: MLGTGPAAATTAATTSSNVSVLQQFASGLKSRNEETRAKAAKELQHYVTM...
Length: 2550

ID: panPan3
Sequence: MLGTGPAAATTAATTSSNVSVLQQFASGLKSRNEETRAKAAKELQHYVTM...
Length: 2550

ID: panTro6
Sequence: MLGTGPAAATTAATTSSNVSVLQQFASGLKSRNEETRAKAAKELQHYVTM...
Length: 2550

ID: gorGor6
Sequence: MLGTGPAAATTAATTSSNVSVLQQFASGLKSRNEETRAKAAKELQHYVTM...
Length: 2550

ID: ponAbe3
Sequence: MLGTGPAAATTAATTSSNVSVLQQFASGLKSRNEETRAKAAKELQHYVTM...
Length: 2550



In [16]:
!pip install legacy-cgi
from ete3 import Tree

tree_path = '/content/drive/MyDrive/joint_asr/MTOR_ox1z4f7_/labelled.nwk'
tree = Tree(tree_path, format=1)

# Print a basic summary of the loaded tree
print("Tree loaded successfully!")
print(f"Total leaves (tips): {len(tree.get_leaves())}")
print(f"Total nodes (including internal): {len(tree.get_descendants()) + 1}")
print("\nFirst 10 leaf names:")
print([leaf.name for leaf in tree.get_leaves()[:10]])

Tree loaded successfully!
Total leaves (tips): 468
Total nodes (including internal): 934

First 10 leaf names:
['hg38', 'panPan3', 'panTro6', 'gorGor6', 'ponAbe3', 'HLhylMol2', 'HLnomLeu4', 'rheMac10', 'HLmacFus1', 'HLmacFas6']


In [18]:

# Map FASTA sequence IDs to their corresponding SeqRecord objects
seq_dict = {record.id: record for record in sequences}

# Map ete3 tree node names to their sequences
node_to_seq = {}
for node in tree.traverse():
    if node.name in seq_dict:
        # Clean sequence by removing gaps '-' and replacing 'X' / invalid chars
        raw_seq = str(seq_dict[node.name].seq)
        cleaned_seq = "".join([amino for amino in raw_seq if amino in "ACDEFGHIKLMNPQRSTVWY"])
        node_to_seq[node.name] = cleaned_seq
    else:
        # Warn if a node's sequence is missing from the FASTA file
        print(f"Warning: No sequence found for node '{node.name}'")

In [ ]:
# Calculate log likelihoods for each node's sequence if they haven't been computed yet
print("Calculating sequence site-wise log likelihoods using VESM...")
node_log_lh = {}
for name, seq in node_to_seq.items():
    print(name)
    # Use the appropriate device (CPU) and pass the tokenizer properly
    node_log_lh[name] = get_LLR(str(seq), model)

# Traversing the tree to calculate the total log likelihood changes along the branches
total_change = 0.0
branch_changes = []

for node in tree.traverse(strategy="preorder"):
    if node.is_root():
        continue

    parent = node.up
    parent_name = parent.name
    child_name = node.name

    if parent_name in node_log_lh and child_name in node_log_lh:
        parent_lh = node_log_lh[parent_name]
        child_lh = node_log_lh[child_name]

        # Compute the sum of log likelihood differences at each site
        # Difference = log_lh(child_residue_at_site_i) - log_lh(parent_residue_at_site_i)
        branch_change = 0.0
        for site in parent_lh:
            if site in child_lh:
                branch_change += (child_lh[site] - parent_lh[site])

        total_change += branch_change
        branch_changes.append((parent_name, child_name, branch_change))

print(f"Completed processing {len(branch_changes)} branches.")
print(f"Sum of VESM log-likelihood changes across all ancestral transitions: {total_change:.4f}")

Calculating sequence site-wise log likelihoods using VESM...
N1
N2
HLtacAcu1
HLornAna3
N3
N449
N4
N432
N450
N465
N5
N162
N433
N444
N451
N463
HLdidVir1
N466
N6
N161
N163
N426
N434
N435
dasNov3
N445
N452
N453
HLthyCyn1
N464
monDom5
HLgraAgi1
N7
N68
tupBel1
tupChi1
N164
N375
HLsolPar1
N427
echTel2
HLmicTal1
eleEdw1
N436
N446
N447
HLphaCin1
HLvomUrs1
N454
N460
HLantFla1
HLsarHar2
N8
HLgalVar2
N69
N155
N165
N302
N376
N408
N428
N429
N437
N438
HLtamTet1
HLmyrTri1
HLchoHof3
N448
N455
N456
HLgymLea1
N461
N9
N51
N70
N139
ochPri3
N156
N166
N290
N303
N370
N377
N395
N409
N421
eriEur2
sorAra2
HLuroGra1
N430
oryAfe1
chrAsi1
N439
N440
HLchoDid2
HLchoDid1
HLphaGym1
HLtriVul1
N457
N458
HLpseOcc1
N462
N10
tarSyr2
N52
N67
N71
N121
N140
N153
N157
N158
N167
N282
N291
N298
N304
N349
N371
N372
N378
N394
N396
HLnocLep1
HLcynBra1
N410
N422
N423
HLscaAqu1
N431
HLhetBru1
HLproCap3
N441
N443
HLmacFul1
HLmacGig1
HLospRuf1
N459
HLpseCor1
HLpseCup1
N11
N39
HLdauMad1
N53
HLnycCou1
otoGar3
N72
HLpedCap1
HLcteGun1
N122


In [8]:
protein_sequence = 'MVTLGVISLLENILVIVAIAKNKLHSPMYFFICSLAVADMLVSVSNGSET'
get_LLR(protein_sequence, model)

,M 1,V 2,T 3,L 4,G 5,V 6,I 7,S 8,L 9,L 10,...,L 41,V 42,S 43,V 44,S 45,N 46,G 47,S 48,E 49,T 50
K,-10.822370,-10.492037,-10.032913,-12.074589,-12.308659,-11.508743,-12.273054,-12.513035,-13.270350,-12.402149,...,-13.256907,-11.394461,-12.387257,-10.438473,-9.992198,-6.134750,-6.267150,-4.923967,-4.186380,-5.011254
R,-10.960470,-10.616225,-10.427553,-11.903950,-11.179328,-11.740171,-12.382093,-11.701326,-12.464409,-12.269700,...,-12.808199,-11.261650,-10.914615,-10.250940,-9.344354,-6.593621,-5.560266,-4.677338,-4.998240,-5.225632
H,-12.340431,-11.180681,-11.048209,-11.962048,-12.962643,-12.276318,-12.853352,-12.572958,-12.915687,-12.686143,...,-12.696779,-11.837661,-12.155983,-10.859957,-10.233177,-6.139750,-6.969483,-5.396991,-5.474028,-5.729684
E,-11.693495,-9.607015,-9.867805,-11.851706,-10.051702,-10.770924,-11.766178,-11.718819,-12.941128,-11.963078,...,-12.862936,-10.560801,-11.349702,-9.709040,-10.128962,-6.586399,-5.456845,-4.734791,0.000000,-5.532530
D,-12.685564,-10.427269,-10.097559,-12.907869,-9.582978,-11.492911,-12.597013,-10.175650,-13.652010,-13.274273,...,-13.956409,-11.139431,-10.490904,-10.355330,-9.688981,-5.478384,-5.690771,-5.119653,-3.711041,-5.610916
N,-11.169938,-9.577674,-8.131983,-10.879612,-10.137532,-10.130110,-11.084190,-8.387291,-12.616671,-11.977329,...,-12.537697,-10.263781,-9.172355,-9.731144,-7.521702,0.000000,-5.597555,-4.286669,-4.639687,-4.267189
Q,-10.726419,-10.365601,-10.238209,-10.482023,-12.357527,-11.571450,-11.984053,-12.804449,-11.777373,-11.491956,...,-11.479870,-10.887246,-12.117641,-10.435312,-10.400277,-6.636483,-6.595416,-4.630557,-4.202579,-5.375794
T,-8.792627,-5.562428,0.000000,-7.565637,-8.550451,-4.820718,-5.862650,-6.848986,-8.170993,-6.186399,...,-9.056640,-7.412440,-6.627876,-5.857553,-5.376298,-4.017410,-4.678298,-3.163659,-4.630419,0.000000
S,-9.889248,-6.644266,-4.023026,-8.432316,-6.165289,-6.079718,-7.489320,0.000000,-8.801502,-7.199619,...,-9.466390,-8.965954,0.000000,-6.816028,0.000000,-3.850564,-3.779921,0.000000,-4.268335,-3.144070
C,-11.727163,-7.902599,-6.022710,-9.327584,-8.278538,-7.084063,-7.935818,-7.311571,-8.864372,-7.208696,...,-9.664170,-9.938976,-7.135559,-7.542041,-6.465246,-6.476341,-5.557200,-4.862798,-6.313693,-5.727869


compute log-likelihood ratio scores (LLR) for all missense mutations with VESM

In [ ]:
get_LLR(protein_sequence, model)

<a id="visualize-results"></a>

## 4. Visualizing Results

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objs as go
from google.colab import output

AAorder=['K','R','H','E','D','N','Q','T','S','C','G','A','V','L','I','M','P','Y','F','W']

def plot_interactive(LLR,higher_than_wt=False,thresh=2,zmax=0,cmap='Viridis_r'):

  TITLE=''

  is_dark = output.eval_js('document.documentElement.matches("[theme=dark]")')
  template='plotly_dark' if is_dark else 'plotly_white'

  fig = px.imshow(LLR.values, x=LLR.columns, y=LLR.index, color_continuous_scale=cmap,zmax=zmax,
                  labels=dict(y="Amino acid change", x="Protein sequence", color="LLR"),
                  template=template,
                  title=TITLE)
  fig.update_xaxes(tickangle=-90,range=[0,99],rangeslider=dict(visible=True),dtick=1)
  fig.update_yaxes(dtick=1)
  fig.update_layout({
  'plot_bgcolor': 'rgba(0, 0, 0, 0)',
  'paper_bgcolor': 'rgba(0, 0, 0, 0)',
  },font={'family':'Arial','size':11},
  hoverlabel=dict(font=dict(family='Arial', size=14)))

  fig.update_traces(
      hovertemplate="<br>".join([
          "<b>%{x} %{y}</b>"+
          " (%{z:.2f})",
      ])+'<extra></extra>'
  )
  if higher_than_wt:
    hwt_x=[]
    hwt_y=[]
    cust=[]
    for i in LLR.columns:
      for j in list(LLR.index[LLR[i]>thresh]):
        hwt_x+=[i]
        hwt_y+=[j]
        cust+=[LLR.loc[j,i]]

    fig.add_trace(go.Scatter(
        x=hwt_x,
        y=hwt_y,
        customdata=cust,
        mode='markers',
        marker=dict(size=8),
        hovertemplate="<br>".join([
            "<b>%{x} %{y}</b>"+
            " (%{customdata:.2f})",
        ])+'<extra></extra>')
    )

  fig.show()


def plot_heatmap(LLR,figname=None,vmin=None):
  """
    Plot a per-residue score heatmap.
      Args:
          scores: 2D array-like of shape (positions, alphabet) or 1D positional scores.
          figname: Optional basename to save PNG.
          range: Color scale range for heatmap.
  """
  primaryLLR=LLR
  plt.figure(figsize=(int(np.round(primaryLLR.shape[1]*90/390)),5))
  sns.heatmap( primaryLLR ,cmap='viridis_r',xticklabels=True, yticklabels=True,vmax=0,vmin=vmin)
  if not figname is None:
    plt.savefig(f"{figname}.png", dpi=300,bbox_inches = 'tight')
  plt.show()

### Visualizing LLRs

In [ ]:
# MC4R
sequence = 'MVNSTHRGMHTSLHLWNRSSYRLHSNASESLGKGYSDGGCYEQLFVSPEVFVTLGVISLLENILVIVAIAKNKNLHSPMYFFICSLAVADMLVSVSNGSETIVITLLNSTDTDAQSFTVNIDNVIDSVICSSLLASICSLLSIAVDRYFTIFYALQYHNIMTVKRVGIIISCIWAACTVSGILFIIYSDSSAVIICLITMFFTMLALMASLYVHMFLMARLHIKRIAVLPGTGAIRQGANMKGAITLTILIGVFVVCWAPFFLHLIFYISCPQNPYCVCFMSHFNLYLILIMCNSIIDPLIYALRSQELRKTFKEIICCYPLGGLCDLSSRY'
plot_interactive(get_LLR(sequence, model))

<a id="visualize_structure"></a>

### Visualize on Structure

In [ ]:
!pip install --quiet py3Dmol biopython

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import py3Dmol
from Bio.SeqUtils import seq1
from Bio.PDB import PDBParser

def show_pdb(position_scores, pdb_file, width=800, height=600, surface=False):
    view = py3Dmol.view(width=width, height=height, js='https://3dmol.org/build/3Dmol.js')
    view.addModel(open(pdb_file, 'r').read(), 'pdb')
    sigmoid_scores = 1 / (1 + np.exp((0.6 * np.array(position_scores) + 6)))
    for i, value in enumerate(sigmoid_scores):
        rgba = plt.cm.coolwarm(value)
        hexcol = matplotlib.colors.rgb2hex(rgba[:3])
        view.setStyle({'resi': str(i+1)}, {'cartoon': {'color': hexcol}})
    view.setBackgroundColor('#383838')

    if surface:
        for i, value in enumerate(sigmoid_scores):
            rgba = plt.cm.coolwarm(value)
            hexcol = matplotlib.colors.rgb2hex(rgba[:3])
            view.addSurface(
                py3Dmol.VDW,
                { 'opacity': 0.7, 'color': hexcol },
                { 'resi': str(i+1) }
            )
        view.setBackgroundColor('white')


    view.zoomTo()
    return view

In [ ]:
import requests
def download_latest_af_pdb(uniprot_id, version_start=20, version_end=1, out_name=None):
    """
    Download the latest AF PDB model version for a UniProt ID.

    Args:
        uniprot_id (str): UniProt accession, e.g., "P12345"
        version_start (int): Highest version to check, e.g., 10
        version_end (int): Lowest version to check, e.g., 1
        out_name (str): Optional output filename. If None, use the server's name.
    """
    base_url = "https://alphafold.ebi.ac.uk/files"

    for v in range(version_start, version_end - 1, -1):
        url = f"{base_url}/AF-{uniprot_id}-F1-model_v{v}.pdb"
        head = requests.head(url)

        if head.status_code == 200:
            print(f"Found version v{v}: {url}")
            response = requests.get(url)
            response.raise_for_status()

            # Determine filename
            filename = out_name or url.split("/")[-1]
            with open(filename, "wb") as f:
                f.write(response.content)
            print(f"Downloaded to: {filename}")
            return filename

    print(f"No model found for UniProt ID '{uniprot_id}' in versions v{version_start}–v{version_end}.")
    return None

Example structure from AFDB

In [ ]:
uniprot_id = 'P32245'
pdb_file = download_latest_af_pdb(uniprot_id, version_start=10, version_end=4, out_name=None)

extract sequence from pdb

In [ ]:
sequence = "".join(
    seq1(res.get_resname())
    for res in PDBParser(QUIET=True).get_structure("model", pdb_file).get_residues()
    if res.id[0] == " "
)

average scores per position

In [ ]:
position_scores = get_LLR(sequence, model).values.mean(0)
show_pdb(position_scores,pdb_file)

<a id="download"></a>
## Download Prediction Scores

- **sequence**: input any amino acid sequence for LLR inference.
- **seq_name**: name the sequence for the file name (optional).
- **heatmap**: check if you want to download the LLR heatmap

In [ ]:
sequence = 'MVNSTHRGMHTSLHLWNRSSYRLHSNASESLGKGYSDGGCYEQLFVSPEVFVTLGVISLLENILVIVAIAKNKNLHSPMYFFICSLAVADMLVSVSNGSETIVITLLNSTDTDAQSFTVNIDNVIDSVICSSLLASICSLLSIAVDRYFTIFYALQYHNIMTVKRVGIIISCIWAACTVSGILFIIYSDSSAVIICLITMFFTMLALMASLYVHMFLMARLHIKRIAVLPGTGAIRQGANMKGAITLTILIGVFVVCWAPFFLHLIFYISCPQNPYCVCFMSHFNLYLILIMCNSIIDPLIYALRSQELRKTFKEIICCYPLGGLCDLSSRY' #@param {type:"string"}
seq_name = "" #@param {type:"string"}
heatmap = True #@param {type:"boolean"}

import hashlib, zipfile
from google.colab import files

def short_hash(seq, length=8):
    return hashlib.sha1(seq.encode()).hexdigest()[:length]

def meltLLR(LLR,gene_prefix=None,ignore_pos=False):
  vars = LLR.melt(ignore_index=False)
  vars['variant'] = [''.join(i.split(' '))+j for i,j in zip(vars['variable'],vars.index)]
  vars['score'] = vars['value']
  vars = vars.set_index('variant')
  if not ignore_pos:
    vars['pos'] = [int(i[1:-1]) for i in vars.index]
  del vars['variable'],vars['value']
  if gene_prefix is not None:
    vars.index=gene_prefix+'_'+vars.index
  return vars

base_name = f'vesm_LLR_{seq_name if len(seq_name) > 0 else short_hash(sequence)}'
csv_file = f"{base_name}.csv"
LLR_scores = get_LLR(sequence, model)

# save csv file
meltLLR(LLR_scores).to_csv(csv_file)

if heatmap:
    png_file = f"{base_name}.png"
    plot_heatmap(LLR_scores, figname=base_name, vmin=None)

    # Create results zip with both files
    zip_name = f"{base_name}_results.zip"
    with zipfile.ZipFile(zip_name, 'w') as zipf:
        zipf.write(csv_file)
        zipf.write(png_file)

file_name = zip_name if heatmap else csv_file

print(f"Saved results to: {file_name}")
files.download(file_name)

<a id="VESM3"></a>

# VESM3 Inference

Downloading the base ESM3-open model requires huggingface login

In [ ]:
from huggingface_hub import login
login()

In [ ]:
!pip install --quiet esm

remove previous model (necessary if using Colab's T4 GPU)

In [ ]:
if 'model' in globals():
    del model
    torch.cuda.empty_cache()
    import gc; gc.collect()

Load VESM3 checkpoint

In [ ]:
vesm3, _ = load_vesm("VESM3", local_dir=local_dir, device=device)

In [ ]:
def get_vesm3_LLR_sequence(sequence, esm3_model):
  from esm.sdk.api import ESMProtein
  protein = ESMProtein(sequence=sequence)
  tokens = esm3_model.encode(protein).sequence
  with torch.no_grad():
    logits =torch.log_softmax(esm3_model.forward(sequence_tokens=tokens.reshape(1,-1)).sequence_logits[0, :, :], dim=-1).cpu()
  tok = tokens.cpu()
  wt_norm = logits[np.arange(len(tok)), tok].unsqueeze(1)
  LLR = logits - wt_norm

  AAorder=['K','R','H','E','D','N','Q','T','S','C','G','A','V','L','I','M','P','Y','F','W']
  order = [esm3_model.tokenizers.sequence.vocab[x] for x in AAorder]
  LLR=LLR[1:-1,order].numpy()
  LLR_ = pd.DataFrame(LLR,columns=AAorder,index=list(sequence)).T
  LLR_.columns = [j.split('.')[0]+' '+str(i+1) for i,j in enumerate(LLR_.columns)]
  return LLR_


def get_vesm3_LLR_structure(pdb_file, esm3_model):
  from esm.sdk.api import ESMProtein
  protein = ESMProtein.from_pdb(pdb_file)
  tokens = esm3_model.encode(protein)
  seq_tokens = tokens.sequence
  struct_tokens = tokens.structure

  with torch.no_grad():
    logits =torch.log_softmax(esm3_model.forward(sequence_tokens=seq_tokens.reshape(1,-1), structure_tokens=struct_tokens.reshape(1,-1) ).sequence_logits[0, :, :], dim=-1).cpu()
  tok = seq_tokens.cpu()
  wt_norm = logits[np.arange(len(tok)), tok].unsqueeze(1)
  LLR = logits - wt_norm

  AAorder=['K','R','H','E','D','N','Q','T','S','C','G','A','V','L','I','M','P','Y','F','W']
  order = [esm3_model.tokenizers.sequence.vocab[x] for x in AAorder]
  LLR=LLR[1:-1,order].numpy()
  LLR_ = pd.DataFrame(LLR,columns=AAorder,index=list(sequence)).T
  LLR_.columns = [j.split('.')[0]+' '+str(i+1) for i,j in enumerate(LLR_.columns)]
  return LLR_

In [ ]:
### Example structure from AFDB
uniprot_id = 'P32245'
pdb_file = download_latest_af_pdb(uniprot_id, version_start=10, version_end=1, out_name=None)

In [ ]:
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
# extract sequence from pdb
sequence = "".join(
    seq1(res.get_resname())
    for res in PDBParser(QUIET=True).get_structure("model", pdb_file).get_residues()
    if res.id[0] == " "
)

### Inference with sequence only

In [ ]:
LLR_seq = get_vesm3_LLR_sequence(sequence, vesm3)

### Inference with structure

In [ ]:
LLR_struct = get_vesm3_LLR_structure(pdb_file, vesm3)

### Visualize Results

Visualize differences between sequence- and structure- derived LLRs with VESM3

In [ ]:
# LLR_seq vs LLR_struct
import matplotlib
import matplotlib.pyplot as plt
plt.plot(LLR_seq.values.reshape(-1), LLR_struct.values.reshape(-1), '.')
plt.plot([-20, 5], [-20, 5], 'k--')
plt.xlabel('LLR from Sequence')
plt.ylabel('LLR from Structure')
plt.grid(True)
plt.show()

Plot the heatmap of differences

In [ ]:
def plot_diff_heatmap(diff,figname=None,range=4):
  plt.figure(figsize=(int(np.round(diff.shape[1]*90/390)),5))
  sns.heatmap( diff ,cmap='coolwarm_r',xticklabels=True, yticklabels=True,vmax=range,vmin=-range)
  if not figname is None:
    plt.savefig(f"{figname}.png", dpi=300,bbox_inches = 'tight')
  plt.show()

diff = LLR_struct - LLR_seq
plot_heatmap(LLR_seq)
threshold = -5
plot_diff_heatmap(diff[((LLR_struct<threshold) & (LLR_seq>threshold)) | ((LLR_struct>threshold) & (LLR_seq<threshold))])
plot_heatmap(LLR_struct)